In [ ]:
# Cell 1 — Imports and setup
# Run this first. If any import fails:
#   uv add pandas matplotlib ipykernel
# Then restart the kernel and re-run.

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# ── Make sure the project root is on sys.path so src.* imports work ──────────
PROJECT_ROOT = Path("..").resolve()   # notebooks/ is one level below root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Plotting defaults ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":      120,
    "figure.figsize":  (9, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize":  13,
    "axes.titleweight": "bold",
    "axes.labelsize":  11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "font.family":     "sans-serif",
})

ACCENT   = "#4C72B0"   # single-series bars / histogram
PALETTE  = [           # multi-series (emotions, languages)
    "#4C72B0", "#DD8452", "#55A868", "#C44E52",
    "#8172B3", "#937860", "#DA8BC3", "#8C8C8C",
    "#CCB974", "#64B5CD", "#4EA7A0", "#B47CC7",
]

MANIFEST_PATH = PROJECT_ROOT / "data" / "manifest.json"

print(f"Project root : {PROJECT_ROOT}")
print(f"Manifest path: {MANIFEST_PATH}")
print(f"Manifest exists: {MANIFEST_PATH.exists()}")

In [ ]:
# Cell 2 — Load manifest → flat DataFrame
#
# Flattens the nested manifest structure into one row per clip so every
# subsequent cell can use standard pandas operations without re-parsing JSON.

def load_manifest_df(path: Path) -> pd.DataFrame:
    """
    Load data/manifest.json and flatten to a tidy DataFrame.

    Columns produced (subset used by charts below):
        id, language, source_type, source_url, source_title
        duration_sec, snr_db, silence_ratio, clipping_pct, quality_score
        emotion, style, emotion_confidence, needs_manual_review
        transcript (raw ASR)
        transcript_corrected, emotion_corrected   ← manual corrections
        verdict, listened, notes                  ← manual review
        dl_passed, pre_passed, seg_passed, qc_passed,
        tx_passed, di_passed, em_passed            ← stage pass flags
        reject_reasons                             ← from quality_check
        speaker_count
    """
    raw = json.loads(path.read_text(encoding="utf-8"))

    # Manifest stored as a list of records
    if isinstance(raw, dict) and "clips" in raw:
        records = raw["clips"]
    elif isinstance(raw, list):
        records = raw
    else:
        raise ValueError(f"Unexpected manifest shape: {type(raw)}")

    rows = []
    for rec in records:
        src   = rec.get("source_info", {})
        stg   = rec.get("stages", {})
        mr    = rec.get("manual_review", {})

        qc    = stg.get("quality_check", {})
        tx    = stg.get("transcription", {})
        di    = stg.get("diarization", {})
        em    = stg.get("emotion_tagging", {})
        seg   = stg.get("segment", {})
        dl    = stg.get("download", {})
        pre   = stg.get("preprocess", {})

        rows.append({
            # ── Identity ──────────────────────────────────────────────────
            "id":                   rec.get("id"),
            "created_at":           rec.get("created_at", "")[:10],

            # ── Source info ───────────────────────────────────────────────
            "language":             src.get("language")    or "unknown",
            "source_type":          src.get("source_type") or "unknown",
            "source_url":           src.get("url")         or "",
            "source_title":         src.get("source_title") or "",

            # ── Segment / duration ────────────────────────────────────────
            "duration_sec":         float(seg.get("duration_sec") or 0),

            # ── Quality metrics ───────────────────────────────────────────
            "snr_db":               float(qc.get("snr_db")         or 0),
            "silence_ratio":        float(qc.get("silence_ratio")  or 0),
            "clipping_pct":         float(qc.get("clipping_pct")   or 0),
            "quality_score":        float(qc.get("quality_score")  or 0),
            "reject_reasons":       "|".join(qc.get("reject_reasons") or []),

            # ── Transcription ─────────────────────────────────────────────
            "transcript":           tx.get("transcript") or "",

            # ── Diarization ───────────────────────────────────────────────
            "speaker_count":        int(di.get("speaker_count") or 1),

            # ── Emotion tagging ───────────────────────────────────────────
            "emotion":              em.get("emotion")    or "unknown",
            "style":                em.get("style")      or "unknown",
            "emotion_confidence":   float(em.get("confidence") or 0),
            "needs_manual_review":  bool(em.get("needs_manual_review")),

            # ── Manual review ─────────────────────────────────────────────
            "listened":             bool(mr.get("listened")),
            "transcript_corrected": mr.get("transcript_corrected") or "",
            "emotion_corrected":    mr.get("emotion_corrected")    or "",
            "notes":                mr.get("notes")                or "",
            "verdict":              mr.get("verdict"),    # None / keep / reject / needs_fix

            # ── Stage pass flags ──────────────────────────────────────────
            "dl_passed":   dl.get("passed"),
            "pre_passed":  pre.get("passed"),
            "seg_passed":  seg.get("passed"),
            "qc_passed":   qc.get("passed"),
            "tx_passed":   tx.get("passed"),
            "di_passed":   di.get("passed"),
            "em_passed":   em.get("passed"),
        })

    df = pd.DataFrame(rows)

    # ── Derived convenience columns ───────────────────────────────────────────
    df["verdict_label"] = df["verdict"].fillna("unreviewed")

    # Use corrected transcript/emotion where available, else pipeline output
    df["final_transcript"] = df["transcript_corrected"].where(
        df["transcript_corrected"] != "", df["transcript"]
    )
    df["final_emotion"] = df["emotion_corrected"].where(
        df["emotion_corrected"] != "", df["emotion"]
    )

    # Word count proxy from final transcript
    df["word_count"] = df["final_transcript"].str.split().str.len().fillna(0).astype(int)

    return df


df = load_manifest_df(MANIFEST_PATH)

print(f"Loaded {len(df)} clips from manifest.\n")
print("Verdict breakdown:")
print(df["verdict_label"].value_counts().to_string())
print(f"\nLanguage breakdown:")
print(df["language"].value_counts().to_string())
print(f"\nStage pass rates:")
for col in ["dl_passed","pre_passed","seg_passed","qc_passed","tx_passed","di_passed","em_passed"]:
    passed = df[col].sum()
    total  = df[col].notna().sum()
    label  = col.replace("_passed","").upper()
    print(f"  {label:12s}: {passed}/{total}")

In [ ]:
# Cell 3 — Duration histogram
#
# Shows whether clips are landing in the target 15–30s window.
# Bars outside the shaded region are worth investigating — very short
# clips may have been cut mid-word; very long ones may need re-splitting.

TARGET_MIN = 15
TARGET_MAX = 30

fig, ax = plt.subplots()

durations = df["duration_sec"].dropna()
durations = durations[durations > 0]   # drop zero-duration placeholder rows

bins = range(
    int(durations.min()) - 1,
    int(durations.max()) + 3,
    1   # 1-second bins
)

ax.hist(durations, bins=bins, color=ACCENT, edgecolor="white", linewidth=0.6)

# Shade the acceptable window
ax.axvspan(TARGET_MIN, TARGET_MAX, alpha=0.12, color="green", label=f"Target window ({TARGET_MIN}–{TARGET_MAX}s)")
ax.axvline(TARGET_MIN, color="green", linestyle="--", linewidth=1.0)
ax.axvline(TARGET_MAX, color="green", linestyle="--", linewidth=1.0)

# Annotate mean and median
ax.axvline(durations.mean(),   color="#C44E52", linestyle="-",  linewidth=1.2, label=f"Mean {durations.mean():.1f}s")
ax.axvline(durations.median(), color="#DD8452", linestyle="--", linewidth=1.2, label=f"Median {durations.median():.1f}s")

ax.set_title("Clip Duration Distribution")
ax.set_xlabel("Duration (seconds)")
ax.set_ylabel("Number of clips")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(fontsize=9)

# Annotate out-of-window clips
n_short = (durations <  TARGET_MIN).sum()
n_long  = (durations >  TARGET_MAX).sum()
n_ok    = len(durations) - n_short - n_long
ax.text(
    0.98, 0.95,
    f"In window: {n_ok}  |  Too short: {n_short}  |  Too long: {n_long}",
    transform=ax.transAxes, ha="right", va="top",
    fontsize=9, color="#555",
)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "chart_duration.png", bbox_inches="tight")
plt.show()
print(f"Clips in target window ({TARGET_MIN}–{TARGET_MAX}s): {n_ok}/{len(durations)}")
print(f"Too short (<{TARGET_MIN}s): {n_short}   Too long (>{TARGET_MAX}s): {n_long}")

In [ ]:
# Cell 4 — SNR distribution histogram
#
# Clips below the threshold (default 20 dB) should have been rejected by
# quality_checker.py. Any that appear here with verdict="keep" are worth
# re-listening to — either the threshold was loosened or the SNR estimate
# is unreliable for that clip type.

SNR_THRESHOLD = 20.0   # should match config.yaml

fig, ax = plt.subplots()

snr_all    = df["snr_db"].dropna()
snr_kept   = df.loc[df["verdict_label"] == "keep",       "snr_db"].dropna()
snr_reject = df.loc[df["verdict_label"] == "reject",     "snr_db"].dropna()
snr_fix    = df.loc[df["verdict_label"] == "needs_fix",  "snr_db"].dropna()

bins = [x * 2 for x in range(
    int(snr_all.min() / 2),
    int(snr_all.max() / 2) + 2
)]  # 2 dB bins

ax.hist(snr_all,    bins=bins, color=ACCENT,    alpha=0.85, label="All clips",   edgecolor="white", lw=0.5)
ax.hist(snr_kept,   bins=bins, color="#55A868", alpha=0.70, label="Kept",        edgecolor="white", lw=0.5)
ax.hist(snr_reject, bins=bins, color="#C44E52", alpha=0.70, label="Rejected",    edgecolor="white", lw=0.5)
ax.hist(snr_fix,    bins=bins, color="#DD8452", alpha=0.70, label="Needs fix",   edgecolor="white", lw=0.5)

ax.axvline(SNR_THRESHOLD, color="red", linestyle="--", linewidth=1.4,
           label=f"Threshold ({SNR_THRESHOLD} dB)")

ax.set_title("SNR Distribution by Verdict")
ax.set_xlabel("Estimated SNR (dB)")
ax.set_ylabel("Number of clips")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(fontsize=9)

# Flag kept clips below threshold — these need explanation in the report
kept_below = df[(df["verdict_label"] == "keep") & (df["snr_db"] < SNR_THRESHOLD)]
if len(kept_below):
    ax.text(
        0.02, 0.95,
        f"⚠ {len(kept_below)} kept clip(s) below SNR threshold — check these",
        transform=ax.transAxes, ha="left", va="top",
        fontsize=9, color="#C44E52",
    )
    print(f"\n⚠ Kept clips below SNR threshold ({SNR_THRESHOLD} dB):")
    print(kept_below[["id","snr_db","notes"]].to_string(index=False))
else:
    print(f"✅ No kept clips below SNR threshold ({SNR_THRESHOLD} dB).")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "chart_snr.png", bbox_inches="tight")
plt.show()

print(f"\nSNR summary across all clips:")
print(snr_all.describe().round(2).to_string())

In [ ]:
# Cell 5 — Emotion tag distribution (final emotion after manual correction)
#
# Uses final_emotion (corrected > pipeline fallback) so the chart reflects
# the dataset as it will actually be uploaded, not the raw LLM output.
# A taxonomy that's heavily skewed toward one tag suggests your source
# selection was too narrow — worth noting in the report.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: all clips ───────────────────────────────────────────────────────────
emotion_all = df["final_emotion"].value_counts()
axes[0].barh(
    emotion_all.index[::-1],
    emotion_all.values[::-1],
    color=PALETTE[:len(emotion_all)],
    edgecolor="white", linewidth=0.5,
)
axes[0].set_title("Emotion Tags — All Clips")
axes[0].set_xlabel("Number of clips")
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

for i, (val, idx) in enumerate(zip(emotion_all.values[::-1], emotion_all.index[::-1])):
    axes[0].text(val + 0.05, i, str(val), va="center", fontsize=9)

# ── Right: kept clips only ────────────────────────────────────────────────────
kept_df = df[df["verdict_label"] == "keep"]
if len(kept_df):
    emotion_kept = kept_df["final_emotion"].value_counts()
    axes[1].barh(
        emotion_kept.index[::-1],
        emotion_kept.values[::-1],
        color=PALETTE[:len(emotion_kept)],
        edgecolor="white", linewidth=0.5,
    )
    axes[1].set_title(f"Emotion Tags — Kept Clips Only (n={len(kept_df)})")
    axes[1].set_xlabel("Number of clips")
    axes[1].xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    for i, (val, idx) in enumerate(zip(emotion_kept.values[::-1], emotion_kept.index[::-1])):
        axes[1].text(val + 0.05, i, str(val), va="center", fontsize=9)
else:
    axes[1].text(0.5, 0.5, "No kept clips yet\n(complete the audit first)",
                 ha="center", va="center", transform=axes[1].transAxes,
                 fontsize=11, color="gray")
    axes[1].set_title("Emotion Tags — Kept Clips Only")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "chart_emotion.png", bbox_inches="tight")
plt.show()

# Print the confidence distribution for uncertain/low-confidence tags
print("\nEmotion confidence summary (all clips):")
print(df.groupby("final_emotion")["emotion_confidence"].describe().round(3).to_string())
print(f"\nClips tagged 'uncertain': {(df['final_emotion'] == 'uncertain').sum()}")
print(f"Clips with confidence < 0.5: {(df['emotion_confidence'] < 0.5).sum()}")

In [ ]:
# Cell 6 — Language balance (pie + grouped bar by verdict)
#
# The pie shows overall split. The grouped bar shows whether one language
# has a much higher rejection rate than the other — which would suggest
# a sourcing problem (e.g. noisier Hindi sources) worth addressing.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: pie chart of all clips by language ──────────────────────────────────
lang_counts = df["language"].value_counts()
wedge_colors = PALETTE[:len(lang_counts)]

axes[0].pie(
    lang_counts.values,
    labels=lang_counts.index,
    autopct="%1.1f%%",
    colors=wedge_colors,
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    textprops={"fontsize": 11},
)
axes[0].set_title(f"Language Balance — All Clips (n={len(df)})")

# ── Right: stacked bar — verdict breakdown per language ───────────────────────
verdict_order  = ["keep", "needs_fix", "reject", "unreviewed"]
verdict_colors = {"keep": "#55A868", "needs_fix": "#DD8452",
                  "reject": "#C44E52", "unreviewed": "#8C8C8C"}

pivot = (
    df.groupby(["language", "verdict_label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=verdict_order, fill_value=0)
)

bottom = pd.Series([0] * len(pivot), index=pivot.index)
for verdict in verdict_order:
    if verdict in pivot.columns:
        axes[1].bar(
            pivot.index,
            pivot[verdict],
            bottom=bottom,
            label=verdict,
            color=verdict_colors[verdict],
            edgecolor="white",
            linewidth=0.5,
        )
        bottom += pivot[verdict]

axes[1].set_title("Verdict Breakdown by Language")
axes[1].set_ylabel("Number of clips")
axes[1].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].legend(fontsize=9, loc="upper right")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "chart_language.png", bbox_inches="tight")
plt.show()

# Duration totals per language
print("\nTotal duration per language (kept clips only):")
kept_dur = df[df["verdict_label"] == "keep"].groupby("language")["duration_sec"].agg(
    clips="count",
    total_sec="sum",
    mean_sec="mean",
)
kept_dur["total_min"] = (kept_dur["total_sec"] / 60).round(1)
kept_dur["mean_sec"]  = kept_dur["mean_sec"].round(1)
print(kept_dur.to_string())

In [ ]:
# Cell 7 — Flagged clips: reject + needs_fix
#
# This is the actionable output of the notebook. Every clip listed here
# should have a documented reason. "No reason recorded" means the manifest
# was not filled in properly — go back to run_audit.py and add notes.
#
# Columns:
#   id          — clip identifier
#   verdict     — reject | needs_fix
#   language    — en-IN | hi-IN
#   snr_db      — was it a quality issue?
#   qc_passed   — did it even pass quality check?
#   reject_reasons — from quality_checker (pipe-separated)
#   emotion     — final emotion tag
#   notes       — your manual audit notes  ← most important column
#   source_type — where the clip came from

flagged = df[df["verdict_label"].isin(["reject", "needs_fix"])].copy()

if flagged.empty:
    print("No rejected or needs_fix clips. Either the audit is not complete,")
    print("or every clip passed — double-check by reviewing manifest.json.")
else:
    display_cols = [
        "id", "verdict_label", "language", "duration_sec",
        "snr_db", "qc_passed", "reject_reasons",
        "final_emotion", "emotion_confidence",
        "speaker_count", "source_type", "notes",
    ]

    # Only keep columns that exist (some may be None-only if pipeline incomplete)
    display_cols = [c for c in display_cols if c in flagged.columns]

    flagged_display = flagged[display_cols].copy()
    flagged_display = flagged_display.rename(columns={
        "verdict_label":       "verdict",
        "final_emotion":       "emotion",
        "emotion_confidence":  "conf",
        "reject_reasons":      "qc_reasons",
    })
    flagged_display["snr_db"]       = flagged_display["snr_db"].round(1)
    flagged_display["duration_sec"] = flagged_display["duration_sec"].round(1)
    flagged_display["conf"]         = flagged_display["conf"].round(2)

    # Highlight rows with no notes — these are under-documented
    no_notes = flagged_display["notes"].isin(["", None]) | flagged_display["notes"].isna()
    n_no_notes = no_notes.sum()

    print(f"Flagged clips: {len(flagged)}  "
          f"({(flagged['verdict_label']=='reject').sum()} rejected, "
          f"{(flagged['verdict_label']=='needs_fix').sum()} needs_fix)")

    if n_no_notes:
        print(f"\n⚠  {n_no_notes} flagged clip(s) have no audit notes — "
              "go back to run_audit.py and document the reason.\n")

    # Render as a styled DataFrame if in Jupyter, else plain print
    try:
        from IPython.display import display as ipy_display

        styled = (
            flagged_display
            .style
            .apply(
                lambda col: [
                    "background-color: #ffeaea" if v == "reject"
                    else "background-color: #fff4e0" if v == "needs_fix"
                    else ""
                    for v in col
                ],
                subset=["verdict"],
            )
            .apply(
                lambda col: [
                    "color: #C44E52; font-weight: bold" if (v == "" or pd.isna(v)) else ""
                    for v in col
                ],
                subset=["notes"],
            )
            .set_table_styles([{
                "selector": "th",
                "props": [("font-size", "11px"), ("text-align", "left")],
            }])
            .format(na_rep="—")
        )
        ipy_display(styled)
    except Exception:
        print(flagged_display.to_string(index=False))

    # Save to CSV for reference in the report
    csv_path = PROJECT_ROOT / "notebooks" / "flagged_clips.csv"
    flagged_display.to_csv(csv_path, index=False)
    print(f"\nSaved to {csv_path}")

In [ ]:
# Cell 8 — Quality score vs SNR scatter
#
# Clips in the bottom-left quadrant (low SNR AND low quality score) that
# somehow got verdict="keep" are the most suspicious — worth a second listen.
# Clips in the top-right (high SNR, high quality score) that got rejected
# might indicate the rejection was based on something other than audio quality
# (e.g. multi-speaker content) — check the reject_reasons column.

fig, ax = plt.subplots(figsize=(8, 5))

verdict_map = {
    "keep":       ("#55A868", "o", 60, "Keep"),
    "reject":     ("#C44E52", "x", 60, "Reject"),
    "needs_fix":  ("#DD8452", "^", 60, "Needs fix"),
    "unreviewed": ("#8C8C8C", "s", 40, "Unreviewed"),
}

for verdict, (color, marker, size, label) in verdict_map.items():
    subset = df[df["verdict_label"] == verdict]
    if subset.empty:
        continue
    ax.scatter(
        subset["snr_db"],
        subset["quality_score"],
        c=color, marker=marker, s=size,
        alpha=0.75, label=f"{label} (n={len(subset)})",
        edgecolors="white" if marker != "x" else color,
        linewidths=0.5,
    )

# Threshold lines
ax.axvline(20, color="red",   linestyle="--", linewidth=1.0, alpha=0.6, label="SNR threshold (20 dB)")
ax.axhline(0.5, color="gray", linestyle=":",  linewidth=1.0, alpha=0.6, label="Quality score 0.5")

# Annotate outliers: kept clips with quality_score < 0.6
outliers = df[(df["verdict_label"] == "keep") & (df["quality_score"] < 0.60)]
for _, row in outliers.iterrows():
    ax.annotate(
        row["id"],
        xy=(row["snr_db"], row["quality_score"]),
        xytext=(4, 4), textcoords="offset points",
        fontsize=7, color="#C44E52",
    )

ax.set_title("Quality Score vs SNR — coloured by verdict")
ax.set_xlabel("Estimated SNR (dB)")
ax.set_ylabel("Composite quality score (0–1)")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "notebooks" / "chart_quality_scatter.png", bbox_inches="tight")
plt.show()

print(f"Kept clips with quality_score < 0.60: {len(outliers)}")
if not outliers.empty:
    print(outliers[["id","snr_db","quality_score","notes"]].to_string(index=False))

In [ ]:
# Cell 9 — Summary stats table
#
# Produces clean numbers ready to paste directly into pipeline_report.md.
# Nothing here should require explanation — if a number surprises you,
# investigate before writing it into the report.

kept = df[df["verdict_label"] == "keep"]

summary = {
    "Total clips in manifest":          len(df),
    "Kept":                             len(kept),
    "Rejected":                         (df["verdict_label"] == "reject").sum(),
    "Needs fix":                        (df["verdict_label"] == "needs_fix").sum(),
    "Unreviewed":                       (df["verdict_label"] == "unreviewed").sum(),
    "Manually listened":                df["listened"].sum(),
    "":                                 "",   # spacer
    "Total duration — all (min)":       f"{df['duration_sec'].sum() / 60:.1f}",
    "Total duration — kept (min)":      f"{kept['duration_sec'].sum() / 60:.1f}",
    "Mean clip duration — kept (sec)":  f"{kept['duration_sec'].mean():.1f}",
    "Median clip duration — kept (sec)":f"{kept['duration_sec'].median():.1f}",
    " ":                                "",   # spacer
    "Mean SNR — all clips (dB)":        f"{df['snr_db'].mean():.1f}",
    "Mean SNR — kept clips (dB)":       f"{kept['snr_db'].mean():.1f}",
    "Clips below SNR threshold":        (df["snr_db"] < 20).sum(),
    "  ":                               "",   # spacer
    "Emotion tags used (kept clips)":   kept["final_emotion"].nunique(),
    "Most common emotion (kept)":       kept["final_emotion"].mode().iloc[0] if len(kept) else "—",
    "Clips tagged 'uncertain'":         (df["final_emotion"] == "uncertain").sum(),
    "Clips needing manual review flag": df["needs_manual_review"].sum(),
    "   ":                              "",   # spacer
    "en-IN clips (kept)":              (kept["language"] == "en-IN").sum(),
    "hi-IN clips (kept)":              (kept["language"] == "hi-IN").sum(),
}

print("=" * 52)
print("  PIPELINE SUMMARY — paste into pipeline_report.md")
print("=" * 52)
for k, v in summary.items():
    if v == "":
        print()
    else:
        print(f"  {k:<42} {v}")
print("=" * 52)

# Also write to a text file for easy copy-paste
summary_path = PROJECT_ROOT / "notebooks" / "summary_stats.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        if v == "":
            f.write("\n")
        else:
            f.write(f"{k:<42} {v}\n")
print(f"\nAlso saved to {summary_path}")